# 02 — Eval sample (cost + quality gate)

Score a stratified **1,000**-message sample with Gemini 3.1 Flash-Lite.
Compare actual spend to the estimator, then hand-label ~150–200 rows before approving a full run.

**Prerequisites:**
- `data/windows.parquet` from notebook 01
- `GEMINI_API_KEY` in `.env`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sentiment.client import make_client
from sentiment.config import ensure_data_dirs, get_pipeline_config
from sentiment.estimate import estimate_from_windows_df
from sentiment.io import save_sample, write_gold_template
from sentiment.metrics import evaluate_against_gold
from sentiment.sampling import stratified_sample
from sentiment.windows import load_windows_parquet

pd.set_option("display.max_colwidth", 100)
paths = ensure_data_dirs()
cfg = get_pipeline_config()
SAMPLE_N = 1000
GOLD_N = 200
SMOKE_N = 50
print("Model:", cfg.model)

## Load windows + stratified sample

In [ ]:
windows = load_windows_parquet()
sample = stratified_sample(windows, n=SAMPLE_N, random_state=42)
sample_path = save_sample(sample, "eval_1k")
print(f"Sampled {len(sample):,} / {len(windows):,} → {sample_path}")
sample.groupby("channel_name").size().sort_values(ascending=False).head(15)

In [ ]:
est = estimate_from_windows_df(sample, batch_size=cfg.batch_size)
print("Estimated cost for 1k sample:")
est.as_dict()

## Smoke test (50 messages)

Confirm schema parsing and rough $/token before scoring the full 1k.

In [ ]:
client = make_client()
smoke = sample.head(SMOKE_N)
smoke_run = client.score_windows(
    smoke,
    batch_size=cfg.batch_size,
    run_name=None,
    show_progress=True,
)
print("Usage:", smoke_run.usage.as_dict())
smoke_run.dataframe.head()

## Score full 1k sample

Only run this after the smoke test looks sane.

In [ ]:
RUN_NAME = "eval_1k"
eval_run = client.score_windows(
    sample,
    batch_size=cfg.batch_size,
    run_name=RUN_NAME,
    checkpoint_every=1,
    resume=True,
    show_progress=True,
)
preds = eval_run.dataframe
print("Scored:", len(preds))
print("Usage:", eval_run.usage.as_dict())
print("Estimate was:", est.as_dict())

In [ ]:
# Merge predictions back onto sample for review
reviewed = sample.merge(preds, on="message_id", how="left")
print(reviewed["polarity"].value_counts(dropna=False))
print("\nSarcasm rate:", round(reviewed["sarcasm"].mean(), 3))
print("Toxicity:")
print(reviewed["toxicity"].value_counts())
reviewed[["channel_name", "author_name", "content", "polarity", "sarcasm", "toxicity", "rationale"]].head(10)

## Cost gate

Actual live cost should be within ~2× of the estimate. If not, inspect batch size / prompt length before scaling.

In [ ]:
actual = eval_run.usage.cost_live_usd
predicted = est.cost_live_usd
ratio = actual / predicted if predicted else float("inf")
print(f"Actual ${actual:.4f} vs estimate ${predicted:.4f} (ratio {ratio:.2f}x)")
if ratio <= 2.0:
    print("COST GATE: PASS")
else:
    print("COST GATE: FAIL — investigate before full run")

## Gold-label template

Fill `gold_polarity`, `gold_sarcasm`, `gold_toxicity` for ~150–200 rows in the CSV, then re-run the metrics cell.

In [ ]:
gold_path = write_gold_template(reviewed, n=GOLD_N)
print("Gold template:", gold_path)
print("Columns: gold_polarity (positive/negative/neutral/mixed), gold_sarcasm (true/false), gold_toxicity (none/mild/moderate/severe)")

In [ ]:
# After labeling, load and score:
gold_path = paths["gold"] / "gold_labels.csv"
if gold_path.exists():
    gold = pd.read_csv(gold_path)
    labeled = gold["gold_polarity"].astype(str).str.strip().ne("")
    if labeled.any():
        metrics = evaluate_against_gold(preds, gold)
        print(metrics)
    else:
        print("No gold labels filled yet.")
else:
    print("Gold file not found.")

## Optional A/B: Gemini 3.5 Flash on 200 rows

Only if Flash-Lite sarcasm quality looks weak. Flash is pricier — use only if needed.

In [ ]:
RUN_AB = False  # set True to compare
if RUN_AB:
    flash = make_client(model="gemini-3.5-flash")
    ab_sample = sample.head(200)
    ab_run = flash.score_windows(ab_sample, batch_size=cfg.batch_size, run_name="eval_flash_200")
    print(ab_run.usage.as_dict())
    ab_run.dataframe.head()

**Gate checklist before notebook 03:**
1. Cost ratio ≤ 2× estimate
2. Spot-check sarcasm / Discord slang looks reasonable
3. Gold polarity accuracy acceptable (aim ≥ ~0.75 as a soft bar)

If all pass → proceed to `03_full_run.ipynb`.